# Buy Signal Analysis

Add more cells below as needed (one per analyze script).

**Before running:** set `DATA_FOLDER` to the folder containing your
session output CSV files. 

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))


BASE_DIR = Path(r"C:\Git\CandleStateSessionAnalysis\data")
STRATEGY = "MACDTarget"  # "MACDTrail" or "MACDTarget"

DATA_FOLDER = BASE_DIR / STRATEGY
OUTPUT_FOLDER = DATA_FOLDER / "output" 

In [ ]:
import pandas as pd

trade_signals = pd.read_csv(OUTPUT_FOLDER / "trade_signals.csv")
trade_signals["CandleOpen"] = pd.to_datetime(trade_signals["CandleOpen"], utc=True).dt.tz_convert("America/New_York")
trade_signals["SessionStart"] = pd.to_datetime(trade_signals["SessionStart"], utc=True).dt.tz_convert("America/New_York")
                 

positions = pd.read_csv(OUTPUT_FOLDER / "positions.csv")
positions["Opened"] = pd.to_datetime(positions["Opened"], utc=True).dt.tz_convert("America/New_York")
positions["Closed"] = pd.to_datetime(positions["Closed"], utc=True).dt.tz_convert("America/New_York")
positions["TimeHeld"] = pd.to_timedelta((positions["IndexClose"] - positions["IndexOpen"]) * 2, unit="m")

#print(trade_signals.shape, positions.shape)
#trade_signals.dtypes
#trade_signals.head(10)
#trade_signals["Signal"].unique()
#trade_signals["Signal"].value_counts()

buy_trade_signals = trade_signals[trade_signals["Signal"] == "Buy"]
#buy_trade_signals.head(10)
#buy_trade_signals["PositionID"].unique()

print(f"Buy signals with null PositionID: {buy_trade_signals['PositionID'].isna().sum()}")
print(f"Buy signals with PositionID: {buy_trade_signals['PositionID'].notna().sum()}")

buy_trade_signals = buy_trade_signals[buy_trade_signals["PositionID"].notna()]
print(f"Buy signals remaining: {len(buy_trade_signals)}")


In [ ]:

positions_subset = positions[["ID", "AvgPrice", "ClosePrice", "RealizedGain", "TimeHeld"]].rename(columns={"ID": "PositionID"})
buy_trade_signals = buy_trade_signals.merge(positions_subset, on="PositionID", how="left")
buy_trade_signals["IsWin"] = (buy_trade_signals["RealizedGain"] > 0).astype(int)
buy_trade_signals[["Symbol", "PositionID", "RealizedGain", "TimeHeld", "IsWin"]]

trade_signals = trade_signals.merge(positions_subset, on="PositionID", how="left")
trade_signals["Unrealized"] = (trade_signals["Close"] - trade_signals["AvgPrice"]) * trade_signals["Shares"]

In [ ]:
#buy_trade_signals["IsWin"].value_counts()
#buy_trade_signals.dtypes
#list(buy_trade_signals.columns)

candle_cols = [
    'CandleHeight', 'CandleBody', 'CandleDelta', 'CandleBodyDelta',
    'CandleBodyPct', 'UpperWickPct', 'LowerWickPct', 'TrueRange',
    'ATR', 'Volatility', 'VolatilityBps', 'VolumeRatio'
]

trend_cols = [
    'VWAP', 'VWAPDelta', 'VWAPSlope', 
    'MarkovState', 'MarkovStateTrend', 'TrendCount', 'ADX', 'MACDHistogram',
    'SlowSlope', 'SlowLongSlope',
    'FastSlope', 'FastLongSlope', 'SlowDelta', 'FastDelta', 'MovingAverageDelta'
]

#buy_trade_signals[analytic_cols].dtypes

buy_trade_signals[candle_cols + trend_cols + ["IsWin"]].corr()["IsWin"].sort_values(ascending=False)

In [ ]:
buy_trade_signals.boxplot(column="ATR", by="IsWin")


In [ ]:
median_atr = buy_trade_signals["ATR"].median()
buy_trade_signals["AboveMedianATR"] = buy_trade_signals["ATR"] > median_atr
buy_trade_signals.groupby("AboveMedianATR")["IsWin"].mean()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

corr = buy_trade_signals[trend_cols + ["IsWin"]].corr()
plt.figure(figsize=(14, 12))
sns.heatmap(corr, cmap="coolwarm", vmin=-1, vmax=1, annot=True, fmt=".2f", annot_kws={"size": 6})
plt.tight_layout()
plt.show()

In [ ]:
#buy_trade_signals[["SlowMA", "FastMA"]].plot(figsize=(12, 5))
#(buy_trade_signals["SlowMA"] - buy_trade_signals["FastMA"]).describe()
#buy_trade_signals[["SlowMA", "FastMA"]]

#buy_trade_signals[["SlowDelta", "FastDelta", "MovingAverageDelta", "FastSlope", "SlowSlope", "IsWin"]].corr()["IsWin"]
#buy_trade_signals[["ATR", "FastSlope"]].corr()

after_1pm_positions_red = trade_signals[
    (trade_signals["PositionID"].notna()) &
    (trade_signals["Unrealized"] < 0) &
    (trade_signals["CandleOpen"].dt.time > pd.Timestamp("13:00:00").time()) &
    (trade_signals["CandleOpen"].dt.date < pd.Timestamp("2026-07-10").date())
]

positionIDs_red_after_1pm = after_1pm_positions_red["PositionID"].unique()
outcomes = buy_trade_signals[buy_trade_signals["PositionID"].isin(positionIDs_red_after_1pm)][["PositionID", "RealizedGain", "IsWin"]].drop_duplicates()

close_info = positions[positions["ID"].isin(positionIDs_red_after_1pm)][["ID", "Closed", "RealizedGain"]].rename(columns={"ID": "PositionID"})

close_info["CloseType"] = close_info["Closed"].dt.time.apply(
    lambda t: "EOD" if t == pd.Timestamp("15:54:00").time() else "PreEOD"
)

eod_positions = close_info[close_info["CloseType"] == "EOD"]["PositionID"]

eod_trajectory = trade_signals[
    trade_signals["PositionID"].isin(eod_positions)
][["PositionID", "CandleOpen", "Close", "SlowSlope", "FastSlope", "Unrealized"]].sort_values(["PositionID", "CandleOpen"])

eod_trajectory.to_csv(OUTPUT_FOLDER / "eod_trajectory.csv", index=False)



In [ ]:
all_negative = eod_trajectory.groupby("PositionID")["FastSlope"].apply(lambda x: (x < 0).all())
print(f"Positions where FastSlope was negative every candle: {all_negative.sum()} of {len(all_negative)}")

mostly_negative = eod_trajectory.groupby("PositionID")["FastSlope"].apply(lambda x: (x < 0).mean() > 0.5)
print(f"Positions where FastSlope was negative more than half the time: {mostly_negative.sum()} of {len(mostly_negative)}")

finished_positive = eod_trajectory.sort_values("CandleOpen").groupby("PositionID")["FastSlope"].last() > 0
print(f"Positions where FastSlope finished positive: {finished_positive.sum()} of {len(finished_positive)}")

summary = pd.DataFrame({
    "AllNegative": eod_trajectory.groupby("PositionID")["FastSlope"].apply(lambda x: (x < 0).all()),
    "MostlyNegative": eod_trajectory.groupby("PositionID")["FastSlope"].apply(lambda x: (x < 0).mean() > 0.5),
    "FinishedPositive": eod_trajectory.sort_values("CandleOpen").groupby("PositionID")["FastSlope"].last() > 0,
})
summary.sum()

In [ ]:
finished_positive_ids = finished_positive[finished_positive].index
eod_losers_realized = positions[positions["ID"].isin(eod_positions)][["ID", "RealizedGain"]].rename(columns={"ID": "PositionID"})

eod_losers_realized["SlopeFinishedPositive"] = eod_losers_realized["PositionID"].isin(finished_positive_ids)
eod_losers_realized.groupby("SlopeFinishedPositive")["RealizedGain"].describe()

In [ ]:
winners_after_1pm = outcomes[outcomes["IsWin"] == 1]["PositionID"]

winner_trajectory = trade_signals[
    trade_signals["PositionID"].isin(winners_after_1pm)
][["PositionID", "CandleOpen", "FastSlope"]].sort_values(["PositionID", "CandleOpen"])

winner_summary = pd.DataFrame({
    "AllNegative": winner_trajectory.groupby("PositionID")["FastSlope"].apply(lambda x: (x < 0).all()),
    "MostlyNegative": winner_trajectory.groupby("PositionID")["FastSlope"].apply(lambda x: (x < 0).mean() > 0.5),
    "FinishedPositive": winner_trajectory.sort_values("CandleOpen").groupby("PositionID")["FastSlope"].last() > 0,
})
winner_summary.sum()